# ARCHIVED historical Phase 2 notebook (source-only)

**NOT THE FINAL Phase 2 execution notebook. DO NOT USE `Run all`.**

This is a redacted, output-free, archival copy of `Project9_Phase2_CORRECTED_v3.ipynb`.
The original notebook was recorded when M5 was interrupted at epoch 26 and before Test. **Later records establish that M5 resumed, stopped at epoch 36, selected the epoch-21 checkpoint, and that the once-only frozen 929-image Test was already completed.** The historical status claims, old due date, and old expected run order in cells below should NOT be used to characterize the submitted project. This notebook does **not** contain current result artifacts or the latest service implementation.

Frozen Test record from the team's 17 Nov report, distinct from any code execution here: M4 box AP50/AP50-95 = `0.3486949280 / 0.2289738041`; M5 mask AP50/AP50-95 = `0.3404084177 / 0.2107355222`. These are **reported historical values**, not reproduced by this archival notebook.

For the contemporary evidence map, see [`../PHASE2_EVIDENCE_STATUS.md`](../PHASE2_EVIDENCE_STATUS.md).

> Safety: All recorded outputs have been removed. Archive extraction now prompts for the password at runtime instead of embedding it. The first code cell explicitly blocks execution, and cells for training/restarting M5 and re-running the final Test are guarded with exceptions. Source data, saved checkpoints, API keys and personal identifiers are not bundled.


## 1. Mount Google Drive and define paths

Colab `/content` files disappear when the runtime resets. This cell must be run in a live Colab session. Store the experiment configs/checkpoints in Google Drive. A copy of the 7GB ZIP in Drive is **optional** and requires adequate Drive storage.

In [ ]:
raise RuntimeError('ARCHIVED HISTORICAL NOTEBOOK: do not Run all. Use maintained Phase 2 source and current evidence.')

from pathlib import Path
from google.colab import drive
import torch, shutil

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/Project9_Phase2')
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
RUNS = DRIVE_PROJECT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)
M4_RUN_NAME = 'M4_yolo11n_640_grouped_v2_seed42'
M5_RUN_NAME = 'M5_yolo11n_seg_640_grouped_v2_seed42'
M4_BEST = RUNS / M4_RUN_NAME / 'weights/best.pt'
M5_BEST = RUNS / M5_RUN_NAME / 'weights/best.pt'
M5_LAST = RUNS / M5_RUN_NAME / 'weights/last.pt'
WORK = Path('/content/Project9_Phase2')
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = Path('/content/ZeroWaste/splits_final_deblurred')
ARCHIVE = Path('/content/zerowaste-f-final.zip')  # corrected spelling
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available')
print('Source ready:', SOURCE.is_dir())
print('ZIP present locally:', ARCHIVE.is_file())
print('Google Drive project:', DRIVE_PROJECT)
print('Local free GB:', round(shutil.disk_usage('/content').free / (1024**3), 1))
print('Existing M4 checkpoint:', M4_BEST.is_file())
print('Existing M5 checkpoint:', M5_BEST.is_file(), '| resume:', M5_LAST.is_file())


## 2. Restore the original source files (only if missing)

If the runtime restarted and the ZIP is absent, re-download from the official Zenodo record. `wget -c` resumes partial downloads. We **verify MD5 before extracting**. The public dataset password is taken from its official record; don't publish credentials for any private dataset.

**NOTE:** The original notebook's final check erroneously looked for `/content/zerowaste-f.zip`; the actual name was `/content/zerowaste-f-final.zip`. More importantly, *all* `/content` files had disappeared in that runtime.

In [ ]:
import subprocess, hashlib

if not all((SOURCE / s / 'labels.json').is_file() for s in ('train', 'val', 'test')):
    if not ARCHIVE.is_file():
        cached = DRIVE_PROJECT / 'cache' / ARCHIVE.name
        if cached.is_file():
            print('Using ZIP cached on Google Drive:', cached)
            ARCHIVE = cached
        else:
            print('No local/Drive copy found. Downloading public ZeroWaste-f dataset.')
            url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
            subprocess.run(['wget', '-c', '--tries=5', '--timeout=60',
                            '-O', str(ARCHIVE), url], check=True)
    # Resume an incomplete local ZIP instead of failing checksum immediately.
    expected_size = 7518242799
    if ARCHIVE.is_file() and ARCHIVE.stat().st_size < expected_size and ARCHIVE.parent == Path('/content'):
        url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
        subprocess.run(['wget', '-c', '--tries=5', '--timeout=60',
                        '-O', str(ARCHIVE), url], check=True)
    md5 = hashlib.md5()
    with ARCHIVE.open('rb') as fh:
        for part in iter(lambda: fh.read(8 * 1024 * 1024), b''):
            md5.update(part)
    expected_md5 = 'e26e31a58080bca6782dca0e56074c5d'
    print('MD5:', md5.hexdigest())
    assert md5.hexdigest() == expected_md5, 'ZIP integrity mismatch. Stop; do not extract.'
    from getpass import getpass
    archive_password = getpass('Official archive password (not stored): ')
    Path('/content/ZeroWaste').mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', '-P', archive_password,
                    str(ARCHIVE), '-d', '/content/ZeroWaste'], check=True)

assert all((SOURCE / s / 'labels.json').is_file() for s in ('train', 'val', 'test'))
print('SOURCE RESTORED:', SOURCE)
for s in ('train', 'val', 'test'):
    print(s, len(list((SOURCE / s / 'data').glob('*.PNG'))), 'images')

## 3. Build the **missing** `candidate` table from COCO metadata

The uploaded original notebook used `candidate.copy()` in its later split-planning cell without any saved cell defining `candidate`. This self-contained cell reconstructs it using the original JSON and image filenames, including `-2` suffixes. Splits and class names come from the files; no guesses or results are invented.

In [ ]:
import json, re
import pandas as pd
from collections import Counter, defaultdict

CLASS_NAMES = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']
SPLITS = ['train', 'val', 'test']
rows = []
category_lookup = {}

for original_split in SPLITS:
    labels_path = SOURCE / original_split / 'labels.json'
    coco = json.loads(labels_path.read_text(encoding='utf-8-sig'))
    categories = {int(c['id']): c['name'] for c in coco['categories']}
    assert set(categories.values()) == set(CLASS_NAMES), categories
    category_lookup[original_split] = categories
    image_ids = {int(im['id']): im for im in coco['images']}
    annotation_counts = defaultdict(Counter)
    for ann in coco['annotations']:
        image_id = int(ann['image_id'])
        assert image_id in image_ids
        annotation_counts[image_id][categories[int(ann['category_id'])]] += 1

    for image_id, image_info in image_ids.items():
        filename = Path(image_info['file_name']).name
        match = re.fullmatch(r'(.+)_frame_(\d+)(?:-\d+)?', Path(filename).stem, re.I)
        if not match:
            raise ValueError(f'Unparsed filename: {filename}')
        source_path = SOURCE / original_split / 'data' / filename
        if not source_path.is_file():
            raise FileNotFoundError(source_path)
        row = dict(original_split=original_split, image_id=image_id,
                   filename=filename, sequence=match.group(1).zfill(2),
                   frame_number=int(match.group(2)),
                   source_path=str(source_path), annotation_path=str(labels_path))
        for name in CLASS_NAMES:
            row[name] = annotation_counts[image_id][name]
        rows.append(row)

candidate = pd.DataFrame(rows)
assert len(candidate) == 4503, 'Unexpected dataset version/count'
assert not candidate.duplicated(['original_split', 'filename']).any()
print('Source files:', candidate.groupby('original_split').size().to_dict())
print('Unusual names handled:', candidate['filename'].str.contains('-2.PNG', regex=False).sum())
print('Source sequences:', candidate.groupby('original_split')['sequence'].nunique().to_dict())
print('COCO objects:', int(candidate[CLASS_NAMES].sum().sum()))

## 4. Create a sequence-disjoint **candidate** split

This reproduces the conservative logic in the original notebook: keep **all original test images unchanged**; remove test-sequence IDs from training/validation; choose validation sequences only from the remaining non-test pool using class balance. The original run selected sequences `07` and `11`, yielding train 1,851, val 451, test 929, excluded 1,272. Recompute from metadata, do not hardcode those numbers as outcomes.

**Method caveat:** Filename prefix is a *proxy* for a source video/sequence. Verify its meaning in the dataset documentation; prefix-disjoint groups reduce temporal leakage but do not mathematically rule out visual duplicates between distinct prefixes. Do not use test metrics to select a model.

In [ ]:
from itertools import combinations
import hashlib

revised = candidate.copy()
original_test = revised['original_split'].eq('test')
test_sources = set(revised.loc[original_test, 'sequence'])
pool_mask = ~original_test & ~revised['sequence'].isin(test_sources)
pool = revised.loc[pool_mask]
by_source = pool.groupby('sequence')[CLASS_NAMES].sum()
by_source.insert(0, 'images', pool.groupby('sequence').size())
print('=== AVAILABLE NON-TEST SOURCE GROUPS ===')
print(by_source.to_string())

totals = by_source[CLASS_NAMES].sum()
assert (totals > 0).all(), 'Source pool is missing a class'
options = []
for n_sources in (2, 3):
    for group in combinations(sorted(by_source.index), n_sources):
        val_c = by_source.loc[list(group), CLASS_NAMES].sum()
        train_c = totals - val_c
        if not ((val_c > 0) & (train_c > 0)).all():
            continue
        val_n = int(by_source.loc[list(group), 'images'].sum())
        score = abs(val_n / len(pool) - 0.20) + float((val_c / totals - 0.20).abs().mean())
        options.append((score, group))
if not options:
    raise ValueError('No valid grouped split; inspect source distribution')
_, chosen_val_sources = min(options)
revised['split'] = 'excluded'
revised.loc[pool_mask, 'split'] = 'train'
revised.loc[pool_mask & revised['sequence'].isin(chosen_val_sources), 'split'] = 'val'
revised.loc[original_test, 'split'] = 'test'

sequence_sets = {s: set(revised.loc[revised['split'].eq(s), 'sequence']) for s in SPLITS}
assert all(not sequence_sets[a].intersection(sequence_sets[b]) for a,b in combinations(SPLITS,2))
assert set(revised.loc[original_test,'source_path']) == set(revised.loc[revised.split.eq('test'),'source_path'])
assert (revised.loc[revised.split.isin(SPLITS)].groupby('split')[CLASS_NAMES].sum() > 0).all().all()

manifest = WORK / 'audits' / 'phase2_grouped_split.csv'
manifest.parent.mkdir(parents=True, exist_ok=True)
revised.to_csv(manifest, index=False)
sha = hashlib.sha256(manifest.read_bytes()).hexdigest()
protocol_dir = DRIVE_PROJECT / 'data_protocol' / sha[:12]
protocol_dir.mkdir(parents=True, exist_ok=True)
(protocol_dir / 'split_manifest.csv').write_bytes(manifest.read_bytes())
print('Validation groups:', chosen_val_sources)
print(revised.groupby('split').size().to_string())
print(revised.loc[revised.split.isin(SPLITS)].groupby('split')[CLASS_NAMES].sum().to_string())
print('SHA256 manifest:', sha)
print('Manifest saved to Drive:', protocol_dir)
print('No source image moved or removed')

## 5. Convert grouped COCO boxes into YOLO detection labels

Class order is now **0 cardboard, 1 metal, 2 rigid_plastic, 3 soft_plastic**, matching the grouped dataset made in the original notebook (but *not* the earlier ungrouped notebook). Do not mix checkpoints/labels across these class mappings.

Uses **hard links where possible** (falling back to copying) so we avoid duplicating GBs in Colab. Writes to a temporary directory and commits only when complete. Reports clipped boxes explicitly.

In [ ]:
import os, math, shutil, tempfile
from PIL import Image
from pathlib import PurePosixPath

det_dir = WORK / 'zerowaste_yolo_grouped_v2'
active = revised.loc[revised.split.isin(SPLITS)].copy()
active['export_name'] = active['original_split'] + '__' + active['filename']
assert not active.duplicated(['split', 'export_name']).any()

def create_yolo_detection():
    expected_manifest_hash = hashlib.sha256(manifest.read_bytes()).hexdigest()
    if det_dir.is_dir():
        info = json.loads((det_dir / 'build_info.json').read_text())
        assert info['manifest_sha256'] == expected_manifest_hash
        assert info['class_names'] == CLASS_NAMES
    else:
        tmp = Path(tempfile.mkdtemp(prefix='.detection_build_', dir=WORK))
        stats = {s: dict(images=0, boxes=0, clipped=0, background=0,
                         **{c: 0 for c in CLASS_NAMES}) for s in SPLITS}
        cached_annotations = {}
        for s in SPLITS:
            (tmp / 'images' / s).mkdir(parents=True)
            (tmp / 'labels' / s).mkdir(parents=True)
        for n, row in enumerate(active.itertuples(index=False), 1):
            if row.annotation_path not in cached_annotations:
                coco = json.loads(Path(row.annotation_path).read_text(encoding='utf-8-sig'))
                images_by_id = {int(i['id']): i for i in coco['images']}
                classes_by_id = {int(c['id']): c['name'] for c in coco['categories']}
                anns = defaultdict(list)
                for a in coco['annotations']:
                    anns[int(a['image_id'])].append(a)
                cached_annotations[row.annotation_path] = images_by_id, classes_by_id, anns
            images_by_id, classes_by_id, anns = cached_annotations[row.annotation_path]
            source = Path(row.source_path)
            W, H = (int(images_by_id[row.image_id][k]) for k in ('width','height'))
            with Image.open(source) as im:
                assert im.size == (W,H)
            lines = []
            for ann in anns[row.image_id]:
                class_name = classes_by_id[int(ann['category_id'])]
                cls = CLASS_NAMES.index(class_name)
                x, y, w, h = map(float, ann['bbox'])
                assert all(map(math.isfinite, (x,y,w,h))) and w > 0 and h > 0
                x1,y1,x2,y2 = (max(0,min(W,x)),max(0,min(H,y)),
                               max(0,min(W,x+w)),max(0,min(H,y+h)))
                assert x2 > x1 and y2 > y1
                stats[row.split]['clipped'] += int((x1,y1,x2,y2) != (x,y,x+w,y+h))
                xc,yc,bw,bh = (x1+x2)/(2*W),(y1+y2)/(2*H),(x2-x1)/W,(y2-y1)/H
                lines.append(f'{cls} {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}')
                stats[row.split]['boxes'] += 1
                stats[row.split][class_name] += 1
            dest_img = tmp / 'images' / row.split / row.export_name
            try:
                os.link(source, dest_img)
            except OSError:
                shutil.copy2(source, dest_img)
            (tmp / 'labels' / row.split / (Path(row.export_name).stem+'.txt')).write_text(
                '\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
            stats[row.split]['images'] += 1
            stats[row.split]['background'] += int(not lines)
            if n % 500 == 0: print('Prepared', n, '/', len(active))
        for s in SPLITS:
            expected = active.loc[active.split.eq(s),CLASS_NAMES].sum()
            assert all(stats[s][name] == int(expected[name]) for name in CLASS_NAMES)
        yaml_text = (f'path: {det_dir}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n'
                     + ''.join(f'  {i}: {name}\n' for i,name in enumerate(CLASS_NAMES)))
        (tmp / 'data.yaml').write_text(yaml_text)
        (tmp / 'split_manifest.csv').write_bytes(manifest.read_bytes())
        info = dict(manifest_sha256=expected_manifest_hash, class_names=CLASS_NAMES,
                    stats=stats, image_method='hardlink or copy', source_dataset='ZeroWaste-f')
        (tmp / 'build_info.json').write_text(json.dumps(info,indent=2))
        tmp.rename(det_dir)
    for s in SPLITS:
        expected = set(active.loc[active.split.eq(s),'export_name'])
        assert expected == {p.name for p in (det_dir/'images'/s).iterdir()}
        assert {Path(v).stem+'.txt' for v in expected} == {p.name for p in (det_dir/'labels'/s).iterdir()}
    info = json.loads((det_dir / 'build_info.json').read_text())
    print(pd.DataFrame(info['stats']).T.to_string())
    print('Detection data YAML:', det_dir / 'data.yaml')

create_yolo_detection()
shutil.copy2(det_dir/'build_info.json', protocol_dir/'det_build_info.json')

## 6. Visual label verification and the split check

The label inspection below is independent of the earlier notebook and uses the **final** grouped dataset. A sequence-disjoint split is guaranteed by the manifest. Exact/near-duplicate visual checks between distinct prefixes should additionally be recorded in the report; this cell does not claim to prove there are no near-duplicates.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, cls in zip(axes.flat, range(4)):
    chosen = None
    for path in sorted((det_dir / 'labels/train').glob('*.txt')):
        lines = path.read_text().splitlines()
        if any(line.startswith(str(cls)+' ') for line in lines):
            chosen = path
            break
    assert chosen, CLASS_NAMES[cls]
    image_path = det_dir / 'images/train' / (chosen.stem+'.PNG')
    with Image.open(image_path) as im:
        img = im.convert('RGB').copy()
    W,H = img.size
    ax.imshow(img)
    for line in chosen.read_text().splitlines():
        c,xc,yc,bw,bh = map(float,line.split())
        x,y = (xc-bw/2)*W,(yc-bh/2)*H
        ax.add_patch(Rectangle((x,y),bw*W,bh*H, fill=False,edgecolor='red',linewidth=1.5))
        ax.text(x,y, CLASS_NAMES[int(c)], fontsize=7, color='yellow',
                bbox={'facecolor':'black','alpha':.6})
    ax.set_title(f'{CLASS_NAMES[cls]} — {chosen.stem}')
    ax.axis('off')
plt.tight_layout()
plt.savefig(protocol_dir / 'm4_grouped_visual_boxes.png', dpi=130)
plt.show()
print('Saved:', protocol_dir / 'm4_grouped_visual_boxes.png')

## 7. Install training libraries

Run only after the source dataset and YAML are ready. The exact package versions used for real experiments are saved in the next training cell; current installs may differ across Colab sessions.

In [ ]:
%pip -q install ultralytics wandb onnx onnxruntime fastapi uvicorn python-multipart httpx

import ultralytics, wandb
print('Ultralytics:', ultralytics.__version__)
print('W&B:', wandb.__version__)

In [ ]:

import os
import wandb
from google.colab import userdata

api_key = userdata.get("WANDB_API_KEY")

assert api_key and len(api_key) > 20, "Check your W&B secret"

os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"

connected = wandb.login(
    key=api_key,
    host="https://api.forge.coreweave.com",
    relogin=True
)

print("W&B connection:", "SUCCESS" if connected else "FAILED")


## 8. M4 training — **COMPLETED; DON'T RERUN**
The historical output below is the **real run** in your uploaded notebook: 34 epochs, best checkpoint and W&B run previously created. This cell is preserved for traceability and will **refuse to overwrite** an existing Drive run. Use sections 9 onward or the checkpoint. Real W&B history: `https://app.forge.coreweave.com/sultan--almutairi/AI251P-Waste-Phase2/runs/epqdtowi`.


In [ ]:
raise RuntimeError('ARCHIVED code for provenance only; training / resume / frozen Test must not be rerun.')

from ultralytics import YOLO, settings
from importlib.metadata import version
from datetime import datetime, timezone

assert torch.cuda.is_available(), 'Enable GPU in Runtime settings'
assert (det_dir/'data.yaml').is_file()
RUNS = DRIVE_PROJECT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)
RUN_NAME = 'M4_yolo11n_640_grouped_v2_seed42'
RUN_DIR = RUNS / RUN_NAME
if RUN_DIR.exists():
    raise RuntimeError(f'RUN EXISTS: {RUN_DIR}. Use the resume cell below, do not overwrite.')

versions = {k:version(k) for k in ['ultralytics','wandb','torch','torchvision','numpy','pandas','Pillow','PyYAML']}
train_args = dict(data=str(det_dir/'data.yaml'), epochs=50, patience=15,
                  imgsz=640, batch=8, device=0, workers=2, cache=False,
                  optimizer='AdamW', lr0=0.001, seed=42, deterministic=True,
                  amp=True, close_mosaic=10, save=True, save_period=5,
                  project=str(RUNS), name=RUN_NAME, exist_ok=False, plots=True)
configuration = dict(task='M4 detection', model='yolo11n.pt', run_name=RUN_NAME,
                     grouped_split_manifest_sha256=sha, train_args=train_args,
                     versions=versions, date_utc=datetime.now(timezone.utc).isoformat())
settings.update({'wandb': True})
assert wandb.login(), 'W&B login is required for live logging'
with wandb.init(project='AI251P-Waste-Phase2', name=RUN_NAME, config=configuration) as run:
    print('Live W&B URL:', run.url)
    model = YOLO('yolo11n.pt')
    model.train(**train_args)
    run.summary['manifest_sha256'] = sha
assert (RUN_DIR/'weights/best.pt').is_file(), 'No best checkpoint produced'
(RUN_DIR/'experiment.json').write_text(json.dumps(configuration,indent=2))
print('Best detection model:', RUN_DIR/'weights/best.pt')

### Resume M4 after an interruption (do **not** rerun if finished)

Rebuild the source and grouped dataset paths first (cells 1–6), then execute this cell only when `last.pt` exists and training was interrupted.

In [ ]:
raise RuntimeError('ARCHIVED code for provenance only; training / resume / frozen Test must not be rerun.')

# MANUAL RESUME — execute only when needed
RESUME_M4 = False
if RESUME_M4:
    from ultralytics import YOLO
    checkpoint = RUNS / 'M4_yolo11n_640_grouped_v2_seed42' / 'weights/last.pt'
    assert checkpoint.is_file()
    YOLO(str(checkpoint)).train(resume=True)

## 9. M4 validation and operating point (validation only)
**The earlier per-threshold `model.val(..., conf=t)` table was misleading.** Ultralytics `metric.box.p` and `.r` are calculated near the F1-optimal operating point, not necessarily at the `conf` you passed. The corrected sweep below interpolates the precision-confidence and recall-confidence curves at each threshold. Report it as **macro per-class curve estimates** (not pooled, image-weighted counts). Keep low-confidence `mAP` separate.

**Provisional operating point:** `confidence=0.25`, `NMS IoU=0.5`, chosen to favor recall over marginally higher macro-F1 at 0.40. This needs to be stated honestly; at 0.25, estimated macro recall is only ~0.526. Do not claim the system meets a safety-critical conveyor acceptance criterion.


In [ ]:
from ultralytics import YOLO
import numpy as np
import pandas as pd

m4_best = RUNS / M4_RUN_NAME / 'weights/best.pt'
assert m4_best.is_file(), 'M4 checkpoint missing from Drive'
assert (det_dir / 'data.yaml').is_file(), 'Restore sections 1–5 first'
m4 = YOLO(str(m4_best))
m4_eval = DRIVE_PROJECT / 'evaluation' / 'M4'
m4_eval.mkdir(parents=True, exist_ok=True)
base = m4.val(data=str(det_dir/'data.yaml'), split='val', imgsz=640,
              conf=0.001, iou=0.5, plots=True,
              project=str(m4_eval), name='val_map_repro', exist_ok=True)
rows = [dict(class_name=base.names[int(i)], ap50=float(base.box.ap50[j]),
             ap50_95=float(base.box.ap[j]))
        for j, i in enumerate(base.box.ap_class_index)]
per_class = pd.DataFrame(rows)
per_class.to_csv(m4_eval/'per_class_ap_val.csv', index=False)
print('VALIDATION mAP50:', float(base.box.map50))
print('VALIDATION mAP50-95:', float(base.box.map))
print('Per-class AP (VAL):')
print(per_class.round(4).to_string(index=False))


In [ ]:
import numpy as np
import pandas as pd
import json

assert 'base' in globals(), 'Run M4 validation cell above first'
x = np.asarray(base.box.px)
p_curves = np.asarray(base.box.p_curve)
r_curves = np.asarray(base.box.r_curve)
assert p_curves.shape == r_curves.shape and p_curves.shape[1] == len(x)
thresholds = [0.05, 0.10, 0.20, 0.25, 0.40, 0.50]
rows = []
for t in thresholds:
    p = np.array([np.interp(t, x, curve) for curve in p_curves])
    r = np.array([np.interp(t, x, curve) for curve in r_curves])
    f1 = 2*p*r / np.maximum(p+r, 1e-12)
    rows.append(dict(confidence=t, macro_precision=float(p.mean()),
                     macro_recall=float(r.mean()), macro_f1=float(f1.mean())))

sweep_df = pd.DataFrame(rows)
sweep_file = m4_eval / 'confidence_sweep_corrected.csv'
sweep_df.to_csv(sweep_file, index=False)
M4_CONF, M4_IOU = 0.25, 0.50
operating_point = dict(split='val', confidence=M4_CONF, nms_iou=M4_IOU,
    source='interpolated Ultralytics P/R curves at IoU 0.5',
    criterion='preliminary recall-prioritized threshold, not max macro F1',
    limitation='macro per-class estimates; not pooled counts')
(m4_eval/'operating_point_provisional.json').write_text(json.dumps(operating_point,indent=2))
print(sweep_df.round(4).to_string(index=False))
print('Saved corrected sweep to:', sweep_file)


## 10. M5 — Build instance segmentation labels from COCO polygons

We use the **same grouped manifest** as M4, but produce a separate YOLO segmentation dataset. COCO `segmentation` polygons are not the same as the grayscale `sem_seg` visualization. Multiple polygons from one object are joined with Ultralytics' utility; **do not invent masks from boxes**. This cell explicitly stops if an annotation has no valid polygon, rather than silently calling it background.

Validate several polygons visually before training, and note any conversion artifacts or joined components.

In [ ]:
from ultralytics.data.converter import merge_multi_segment
import numpy as np

seg_dir = WORK / 'zerowaste_seg_grouped_v2'
if not seg_dir.is_dir():
    tmp = Path(tempfile.mkdtemp(prefix='.segmentation_build_',dir=WORK))
    for s in SPLITS:
        (tmp/'images'/s).mkdir(parents=True)
        (tmp/'labels'/s).mkdir(parents=True)
    cache = {}
    multi_count = 0
    missing_count = 0
    for index,row in enumerate(active.itertuples(index=False),1):
        if row.annotation_path not in cache:
            coco = json.loads(Path(row.annotation_path).read_text(encoding='utf-8-sig'))
            lookup = {int(i['id']):i for i in coco['images']}
            categories = {int(c['id']):c['name'] for c in coco['categories']}
            all_annotations = defaultdict(list)
            for ann in coco['annotations']:
                all_annotations[int(ann['image_id'])].append(ann)
            cache[row.annotation_path] = (lookup,categories,all_annotations)
        images_by_id,categories,all_annotations = cache[row.annotation_path]
        info = images_by_id[row.image_id]
        W,H = int(info['width']),int(info['height'])
        output = []
        for ann in all_annotations[row.image_id]:
            c = CLASS_NAMES.index(categories[int(ann['category_id'])])
            raw = ann.get('segmentation',[])
            if not isinstance(raw,list):
                raise ValueError(f'RLE annotation requires manual conversion: {ann["id"]}')
            polygons = [p for p in raw if len(p)>=6 and len(p)%2==0
                        and all(math.isfinite(float(v)) for v in p)]
            if not polygons:
                missing_count += 1
                raise ValueError(f'Missing instance polygon: ann={ann["id"]}, image={row.filename}')
            if len(polygons)>1:
                multi_count += 1
                points = np.concatenate(merge_multi_segment(polygons),axis=0).astype(float)
            else:
                points = np.array(polygons[0],dtype=float).reshape(-1,2)
            points[:,0] = np.clip(points[:,0]/W,0,1)
            points[:,1] = np.clip(points[:,1]/H,0,1)
            if len(points)<3:
                raise ValueError(f'Unusable polygon for {ann["id"]}')
            output.append(str(c)+' '+' '.join(f'{v:.8f}' for v in points.reshape(-1)))
        dest_img = tmp/'images'/row.split/row.export_name
        try:
            os.link(Path(row.source_path),dest_img)
        except OSError:
            shutil.copy2(row.source_path,dest_img)
        (tmp/'labels'/row.split/(Path(row.export_name).stem+'.txt')).write_text(
            '\n'.join(output)+('\n' if output else ''),encoding='utf-8')
        if index%500==0: print('Converted segment images:', index,'/',len(active))
    yaml_text = (f'path: {seg_dir}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n'
                 + ''.join(f'  {i}: {name}\n' for i,name in enumerate(CLASS_NAMES)))
    (tmp/'data.yaml').write_text(yaml_text)
    (tmp/'split_manifest.csv').write_bytes(manifest.read_bytes())
    (tmp/'build_info.json').write_text(json.dumps(dict(manifest_sha256=sha,
         multi_polygon_annotations=multi_count, missing_polygon_annotations=missing_count,
         classes=CLASS_NAMES),indent=2))
    tmp.rename(seg_dir)
else:
    info = json.loads((seg_dir/'build_info.json').read_text())
    assert info['manifest_sha256']==sha
print('Segmentation dataset:',seg_dir)
print('Conversion log:',(seg_dir/'build_info.json').read_text())

## 11. M5 polygon visual check

Before training, inspect at least four images from the training split and confirm the colored polygons align with the objects.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from matplotlib.patches import Polygon

# Prefer a different source sequence for each category, instead of four adjacent video frames.
paths = sorted((seg_dir/'labels/train').glob('*.txt'))
selected = []
used_sequences = set()
for cls in range(len(CLASS_NAMES)):
    choices = []
    for label in paths:
        lines = label.read_text().splitlines()
        if any(line.startswith(f'{cls} ') for line in lines):
            match = re.search(r'_frame_', label.stem)
            seq = label.stem[:match.start()] if match else label.stem
            if label not in selected:
                choices.append((seq in used_sequences, label, seq))
    assert choices, f'No sample contains class {CLASS_NAMES[cls]}'
    _, path, seq = sorted(choices, key=lambda t:(t[0],str(t[1])))[0]
    selected.append(path); used_sequences.add(seq)
fig,axes=plt.subplots(2,2,figsize=(14,9))
for ax, label, class_name in zip(axes.flat,selected,CLASS_NAMES):
    image_path = seg_dir/'images/train'/(label.stem+'.PNG')
    with Image.open(image_path) as image:
        W,H = image.size
        ax.imshow(image.convert('RGB'))
    for line in label.read_text().splitlines():
        parts=line.split()
        pts=np.array(list(map(float,parts[1:]))).reshape(-1,2)*[W,H]
        ax.add_patch(Polygon(pts,closed=True,alpha=0.25,edgecolor='red',facecolor='lime'))
    ax.set_title(f'{class_name}: {label.stem}')
    ax.axis('off')
plt.tight_layout()
plt.savefig(protocol_dir/'m5_grouped_visual_polygons_diverse.png',dpi=130)
plt.show()
print('Inspected distinct training frames:', [p.stem for p in selected])


### M5 readiness (run before **new** training or resume)
Use your Colab Secret `WANDB_API_KEY`. The existing run was already started; do **not** repeat the original initial-training cell. If GPU is unavailable, use the CPU-safe progress summary below or M8 export and return to resume after GPU access is restored.


In [ ]:

import os
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"

import torch
import wandb
from google.colab import userdata

key = userdata.get("WANDB_API_KEY")
assert key, "W&B key missing from Secrets"

connected = wandb.login(
    key=key,
    host="https://api.forge.coreweave.com",
    relogin=True
)

print("GPU:", torch.cuda.is_available())
print("Segmentation dataset:", (seg_dir / "data.yaml").is_file())
print("W&B connected:", connected)
print("Drive runs folder:", RUNS.is_dir())


> **HISTORICAL SNAPSHOT ONLY.** M5 later completed early stopping at epoch 36 and the original Test evaluation has been executed. Do not follow the old resume or Test instructions.

## 12. M5 initial training — **INTERRUPTED; DON'T RERUN FROM SCRATCH**
The saved historical output shows it reached epoch **26 of 50**, and model weights/CSV were present on Drive. Keep this cell as training provenance; since the run folder exists, it will refuse to start over. To **resume**, use the dedicated guarded cell below.

W&B run from the actual training output: `https://app.forge.coreweave.com/sultan--almutairi/AI251P-Waste-Phase2/runs/c6kv4slq`.


In [ ]:
raise RuntimeError('ARCHIVED code for provenance only; training / resume / frozen Test must not be rerun.')

from ultralytics import YOLO, settings
settings.update({'wandb':True})
seg_run_name='M5_yolo11n_seg_640_grouped_v2_seed42'
seg_run_dir=RUNS/seg_run_name
if seg_run_dir.exists():
    raise RuntimeError(f'Existing segmentation run: {seg_run_dir}. Resume or use new name.')
assert wandb.login()
seg_train_args=dict(data=str(seg_dir/'data.yaml'),epochs=50,patience=15,
                    imgsz=640,batch=8,device=0,workers=2,seed=42,
                    amp=True,optimizer='AdamW',lr0=0.001,
                    save_period=5,plots=True,project=str(RUNS),
                    name=seg_run_name,exist_ok=False)
seg_config=dict(task='M5 instance segmentation',model='yolo11n-seg.pt',
                manifest_sha256=sha,train_args=seg_train_args)
with wandb.init(project='AI251P-Waste-Phase2',name=seg_run_name,config=seg_config) as run:
    print('Live W&B:',run.url)
    YOLO('yolo11n-seg.pt').train(**seg_train_args)
assert (seg_run_dir/'weights/best.pt').is_file()
(seg_run_dir/'experiment.json').write_text(json.dumps(seg_config,indent=2))
print('Best segmentation model:',seg_run_dir/'weights/best.pt')

### M5 checkpoints and learning curve — **CPU-safe**
This cell works even when Colab reports `GPU available: False`; it reads the real CSV from Drive. The PNG is evidence of the *partial* 26-epoch training only. If Drive is not mounted, run section 1 first. It cannot resume training without GPU.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

run_dir = RUNS / M5_RUN_NAME
assert (run_dir/'weights/best.pt').is_file(), 'Best checkpoint missing'
assert (run_dir/'weights/last.pt').is_file(), 'Last checkpoint missing'
assert (run_dir/'results.csv').is_file(), 'Training CSV missing; check Drive mount'
progress = pd.read_csv(run_dir/'results.csv')
progress.columns = progress.columns.str.strip()
metric = 'metrics/mAP50-95(M)'
best_row = progress.loc[progress[metric].idxmax()]
print('Last completed epoch:', int(progress['epoch'].iloc[-1]))
print('Best validation epoch:', int(best_row['epoch']))
print('Best mask mAP50-95:', round(float(best_row[metric]), 4))
print('GPU available:', torch.cuda.is_available())
plt.figure(figsize=(9,4))
plt.plot(progress['epoch'],progress[metric],marker='.')
plt.title('M5 partial run — validation mask mAP50–95')
plt.xlabel('Epoch'); plt.ylabel('Mask mAP50–95'); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(run_dir/'m5_validation_progress.png',dpi=150)
plt.show()


> **HISTORICAL SNAPSHOT ONLY.** M5 later completed early stopping at epoch 36 and the original Test evaluation has been executed. Do not follow the old resume or Test instructions.

### **RESUME M5** — only after GPU is actually available
1. On a new runtime, execute **1 → 5, 7, 10** (recreate `seg_dir`) and ensure section 7 installed `ultralytics` and section 7's W&B login succeeded. Do **NOT** run the original M5 initial training cell again.
2. Verify `torch.cuda.is_available()` returns **True**. If not, leave `RESUME_M5=False`, and use CPU-only steps instead.
3. Set `RESUME_M5=True` below **once**, run it, and monitor progress. It loads `last.pt` and restores optimizer and epoch state, rather than restarting with `best.pt`.
4. The resumed training may stop early again. Use the saved CSV and validation to report exactly which epochs ran. W&B resumed execution uses a **separate live W&B evidence run**, not fabricated historic metrics.

Ref: [Ultralytics resume documentation](https://docs.ultralytics.com/modes/train/#resuming-interrupted-trainings).


In [ ]:
raise RuntimeError('ARCHIVED code for provenance only; training / resume / frozen Test must not be rerun.')

RESUME_M5 = False  # deliberately disabled until GPU + dataset are ready

if RESUME_M5:
    from ultralytics import YOLO
    from google.colab import userdata
    import os, json, wandb
    assert torch.cuda.is_available(), 'No GPU! Do not resume on CPU'
    assert (seg_dir/'data.yaml').is_file(), 'Run sections 1–5 and 10 to rebuild segmentation data'
    assert M5_LAST.is_file(), 'Cannot find last.pt in Google Drive'
    os.environ['WANDB_BASE_URL'] = 'https://api.forge.coreweave.com'
    key = userdata.get('WANDB_API_KEY')
    assert key, 'Configure Colab Secret WANDB_API_KEY'
    assert wandb.login(key=key, host=os.environ['WANDB_BASE_URL'], relogin=True)
    run_name = M5_RUN_NAME + '_resumed_from_checkpoint'
    with wandb.init(project='AI251P-Waste-Phase2', name=run_name,
                    config={'resume_from':str(M5_LAST), 'dataset':str(seg_dir/'data.yaml'),
                            'manifest_sha256':sha, 'type':'M5 checkpoint resume'}) as run:
        print('Live resumed training W&B run:',run.url)
        # Resume is only legal on an interrupted, non-completed checkpoint.
        # data= overrides any stale /content data path from the checkpoint.
        YOLO(str(M5_LAST)).train(resume=True, data=str(seg_dir/'data.yaml'),
                                 device=0, workers=2)
    print('Resume finished/stopped; inspect results.csv and weights in:', RUNS/M5_RUN_NAME)
else:
    print('M5 resume NOT started. Set RESUME_M5=True only on GPU after restoring data.')


## 13. M5 validation: mask AP and **separate** IoU/Dice audit
Run after restoring dataset; CPU is possible but slower. AP (full Validation) comes from Ultralytics segmentation validation. The mean mask IoU/Dice here is a **small, explicitly labelled subset audit**, computed by matching predictions to a same-class ground-truth polygon. Unmatched ground-truth instances count as zero. Predictions with no match are not included in the Dice average (they are penalized by mask AP). Do not describe this limited audit as full-split test mIoU/Dice. The source COCO polygons were converted to YOLO; merged multi-part masks can differ from original and require manual inspection.


In [ ]:
from ultralytics import YOLO
from PIL import Image
import cv2, numpy as np, pandas as pd
from collections import defaultdict

m5_best = RUNS / M5_RUN_NAME / 'weights/best.pt'
assert m5_best.is_file(), 'M5 best.pt not found'
assert (seg_dir/'data.yaml').is_file(), 'Recreate segmentation dataset using section 10'
m5 = YOLO(str(m5_best))
m5_eval = DRIVE_PROJECT/'evaluation'/'M5'
m5_eval.mkdir(parents=True,exist_ok=True)
seg_metrics = m5.val(data=str(seg_dir/'data.yaml'),split='val',imgsz=640,
                     conf=0.001, plots=True, project=str(m5_eval),
                     name='val_mask_map',exist_ok=True)
print('Full VAL mask mAP50:',float(seg_metrics.seg.map50))
print('Full VAL mask mAP50–95:',float(seg_metrics.seg.map))

# Fixed, diverse validation subset for rasterized per-instance IoU/Dice audit.
ALL_VAL = sorted((seg_dir/'labels/val').glob('*.txt'))
rng = np.random.default_rng(42)
chosen = []
for label in ALL_VAL:
    if label.read_text().strip():
        name=label.stem
        seq=name.split('_frame_')[0]
        if seq not in [x.stem.split('_frame_')[0] for x in chosen]:
            chosen.append(label)
others=[p for p in ALL_VAL if p not in chosen and p.read_text().strip()]
chosen += [others[i] for i in rng.permutation(len(others))[:max(0,24-len(chosen))]]
assert len(chosen)>0, 'No nonempty validation labels'
records=[]
for label in chosen:
    path=seg_dir/'images/val'/(label.stem+'.PNG')
    with Image.open(path) as im:
        W,H=im.size
    ground=[]
    for line in label.read_text().splitlines():
        vals=line.split(); cid=int(vals[0])
        pts=np.asarray([float(v) for v in vals[1:]]).reshape(-1,2)
        pts=np.rint(pts*np.array([W,H])).astype(np.int32)
        mask=np.zeros((H,W),np.uint8);cv2.fillPoly(mask,[pts],1)
        ground.append((cid,mask.astype(bool)))
    pred=m5.predict(str(path),imgsz=640,conf=0.25,iou=0.5,
                    retina_masks=True,verbose=False)[0]
    predictions=[]
    if pred.masks is not None:
        for box,pts in zip(pred.boxes, pred.masks.xy):
            mask=np.zeros((H,W),np.uint8)
            if len(pts)>=3: cv2.fillPoly(mask,[np.rint(pts).astype(np.int32)],1)
            predictions.append((int(box.cls.item()),mask.astype(bool)))
    used=set()
    for cid,gt in ground:
        candidates=[]
        for j,(pc,pm) in enumerate(predictions):
            if pc!=cid or j in used: continue
            inter=np.logical_and(gt,pm).sum(); union=np.logical_or(gt,pm).sum()
            candidates.append((float(inter/union) if union else 0.0,j))
        iou,j=max(candidates,default=(0.0,None))
        if j is not None and iou>0:
            used.add(j)
            pm=predictions[j][1]
            dice=2*np.logical_and(gt,pm).sum()/(gt.sum()+pm.sum()+1e-12)
        else:
            iou,dice=0.0,0.0
        records.append(dict(image=label.name,class_name=CLASS_NAMES[cid],
                            matched=(j is not None and iou>0),iou=float(iou),dice=float(dice)))
audit=pd.DataFrame(records)
audit.to_csv(m5_eval/'mask_iou_dice_VAL_SUBSET24.csv',index=False)
print('Subset:',len(chosen),'images /',len(audit),'GT objects — VALIDATION ONLY')
print('Subset mean IoU:',round(float(audit.iou.mean()),4),
      'mean Dice:',round(float(audit.dice.mean()),4))
print(audit.groupby('class_name')[['iou','dice']].mean().round(4).to_string())


## 14. M8 — ONNX export and **measured** parity (CPU possible)
This is export, not a completed deployed service. Both M4 and M5 PyTorch checkpoints are exported, checked structurally with `onnx.checker`, then compared to ONNX Runtime on a few **validation** frames (not Test). Parity is a sampled operational-point agreement test; record images, confidence and matching IoU. **CPU is allowed**, but can be slower. Don't call sample parity a full-system acceptance result. Original M5 training can be resumed later without invalidating saved earlier validations.


In [ ]:
import json, shutil, hashlib
from pathlib import Path
import numpy as np
from ultralytics import YOLO

assert M4_BEST.is_file() and M5_BEST.is_file(), 'M4/M5 best.pt must exist in Drive'
onnx_dir = DRIVE_PROJECT/'deployment'/'onnx'
onnx_dir.mkdir(parents=True, exist_ok=True)

# On some Colab versions the ONNX exporter installs extra dependencies on first use.
import onnx
exports={}
for task,weights in [('detect',M4_BEST),('segment',M5_BEST)]:
    path=Path(YOLO(str(weights)).export(format='onnx',imgsz=640,opset=17,
                                         dynamic=False,simplify=False,device='cpu'))
    model=onnx.load(str(path));onnx.checker.check_model(model)
    destination=onnx_dir/f'{task}.onnx'
    if path.resolve()!=destination.resolve(): shutil.copy2(path,destination)
    exports[task]=str(destination)
    print(task, 'ONNX valid:',destination,'bytes:',destination.stat().st_size)

# Record export hashes, exact provenance and backend info for release manifest.
def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
export_meta={task:{'path':path,'sha256':sha256_file(path)} for task,path in exports.items()}
(onnx_dir/'exports_sha256.json').write_text(json.dumps(export_meta,indent=2))
print('Export manifests:',onnx_dir/'exports_sha256.json')


In [ ]:
import numpy as np, json
from ultralytics import YOLO

assert (det_dir/'data.yaml').is_file(), 'Restore grouped YOLO data for parity validation'
val_images = sorted((det_dir/'images/val').glob('*.PNG'))[::max(1,len(list((det_dir/'images/val').glob('*.PNG')))//8)][:8]
assert len(val_images)>0

def iou_xyxy(a,b):
    x1=max(a[0],b[0]);y1=max(a[1],b[1]);x2=min(a[2],b[2]);y2=min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1]);ab=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/(aa+ab-inter+1e-12)

# Compare decoded boxes AND class identities. Matching is unique per prediction.
def score_parity(pt,ort,paths,task):
    matched=0; total=0; extra_ort=0
    for p in paths:
        aa=pt.predict(str(p),imgsz=640,conf=0.25,iou=0.5,device='cpu',verbose=False)[0]
        bb=ort.predict(str(p),imgsz=640,conf=0.25,iou=0.5,device='cpu',verbose=False)[0]
        a_boxes=aa.boxes.xyxy.cpu().numpy(); a_cls=aa.boxes.cls.int().cpu().numpy()
        b_boxes=bb.boxes.xyxy.cpu().numpy(); b_cls=bb.boxes.cls.int().cpu().numpy()
        used=set()
        for box,c in zip(a_boxes,a_cls):
            total+=1
            options=[(iou_xyxy(box,other),j) for j,(other,k) in enumerate(zip(b_boxes,b_cls))
                     if j not in used and c==k]
            if options:
                best,j=max(options)
                if best>=0.90: matched+=1;used.add(j)
        extra_ort+=len(b_boxes)-len(used)
    return dict(task=task,split='val',image_count=len(paths),conf=0.25,nms_iou=0.5,
                match_iou=0.90,pytorch_detections=total,matched=matched,
                onnx_unmatched=extra_ort,matched_fraction=matched/total if total else None,
                warning='sampled decoded-box parity only; raw-tensor and mask parity are not yet checked')

parity=[]
for task,weight in [('detect',M4_BEST),('segment',M5_BEST)]:
    parity.append(score_parity(YOLO(str(weight)),YOLO(exports[task]),val_images,task))
(onnx_dir/'sampled_val_parity.json').write_text(json.dumps(parity,indent=2))
print(json.dumps(parity,indent=2))


## 15. M8 — runnable HTTP service + release manifest
This generates files in `Google Drive/Project9_Phase2/deployment/http_service`. It deliberately uses a **manifest** to choose class order, input size, thresholds, backend model checksums and device. `GET /health` and `POST /predict` are endpoints. The application runs PyTorch for both models (ONNX exported separately), matches their masks by class and box overlap, returns a centroid **only** when a segmentation mask matches, and returns per-material counts.

**It is not a remotely hosted, always-on service yet.** Run `uvicorn service:app ...` in a proper environment and send an actual HTTP request before claiming deployment. Benchmark results are specific to the device and service path; no conveyor speed was provided, so throughput compliance cannot yet be determined.


In [ ]:
import os,json,shutil,hashlib
from pathlib import Path
from importlib.metadata import version

assert M4_BEST.is_file() and M5_BEST.is_file(), 'Need model checkpoints in Drive'
service_dir=DRIVE_PROJECT/'deployment'/'http_service'
service_dir.mkdir(parents=True,exist_ok=True)
shutil.copy2(M4_BEST,service_dir/'best_det.pt')
shutil.copy2(M5_BEST,service_dir/'best_seg.pt')

def sha256_path(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()

release={
    'project':'AI251P-Project9-Phase2', 'class_names':CLASS_NAMES,
    'weights':{'detector':{'file':'best_det.pt','sha256':sha256_path(M4_BEST)},
               'segmenter':{'file':'best_seg.pt','sha256':sha256_path(M5_BEST)}},
    'confidence':0.25,'nms_iou':0.50,'imgsz':640,'device':'cpu',
    'backend':'PyTorch','input_format':'RGB upload; converted to BGR numpy for Ultralytics',
    'inference':{'box_mask_match_min_iou':0.50,'max_upload_mb':12},
    'versions':{name:version(name) for name in ['ultralytics','torch','fastapi','numpy']},
    'status':'prototype, not yet externally hosted; M5 training was interrupted'
}
(service_dir/'release_manifest.json').write_text(json.dumps(release,indent=2))
service_source="from pathlib import Path\nimport io,json,hashlib\nimport cv2\nimport numpy as np\nfrom PIL import Image, UnidentifiedImageError\nfrom fastapi import FastAPI, File, UploadFile, HTTPException\nfrom ultralytics import YOLO\n\nHERE=Path(__file__).resolve().parent\nCONFIG=json.loads((HERE/'release_manifest.json').read_text())\nImage.MAX_IMAGE_PIXELS=16_000_000\n\ndef check_hash(path,expected):\n    h=hashlib.sha256()\n    with path.open('rb') as f:\n        for chunk in iter(lambda:f.read(4*1024*1024),b''): h.update(chunk)\n    if h.hexdigest()!=expected: raise RuntimeError('Model checksum mismatch')\n\ndet_path=HERE/CONFIG['weights']['detector']['file']\nseg_path=HERE/CONFIG['weights']['segmenter']['file']\nfor key,path in [('detector',det_path),('segmenter',seg_path)]:\n    if not path.is_file(): raise RuntimeError(f'Missing {path}')\n    check_hash(path,CONFIG['weights'][key]['sha256'])\ndet=YOLO(str(det_path))\nseg=YOLO(str(seg_path))\napp=FastAPI(title='Project9 Waste Conveyor — Phase2 prototype')\n\n@app.get('/health')\ndef health():\n    return {'status':'ok','backend':'PyTorch','device':CONFIG['device'],\n            'classes':CONFIG['class_names'], 'model_verified':True}\n\ndef overlap(a,b):\n    x1=max(a[0],b[0]);y1=max(a[1],b[1]);x2=min(a[2],b[2]);y2=min(a[3],b[3])\n    inter=max(0,x2-x1)*max(0,y2-y1)\n    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1]);ab=max(0,b[2]-b[0])*max(0,b[3]-b[1])\n    return inter/(aa+ab-inter+1e-12)\n\n@app.post('/predict')\nasync def predict(file:UploadFile=File(...)):\n    raw=await file.read(12*1024*1024+1)\n    if len(raw)>12*1024*1024:raise HTTPException(413,'Max upload 12 MB')\n    if not raw:raise HTTPException(400,'Empty file')\n    try:\n        with Image.open(io.BytesIO(raw)) as im:\n            img=im.convert('RGB')\n        if not 64<=img.width<=4096 or not 64<=img.height<=4096:\n            raise ValueError('Image dimensions out of range')\n        rgb=np.array(img)\n        bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)\n    except (UnidentifiedImageError,ValueError,OSError,Image.DecompressionBombError):\n        raise HTTPException(400,'Invalid or unsupported image')\n    opts=dict(conf=CONFIG['confidence'],iou=CONFIG['nms_iou'],imgsz=CONFIG['imgsz'],\n              device=CONFIG['device'],verbose=False)\n    # Ultralytics expects BGR numpy arrays; upload RGB is converted above.\n    detection=det.predict(bgr,**opts)[0]\n    segmentation=seg.predict(bgr,retina_masks=True,**opts)[0]\n    d_boxes=detection.boxes.xyxy.cpu().numpy()\n    d_classes=detection.boxes.cls.int().cpu().numpy()\n    d_scores=detection.boxes.conf.cpu().numpy()\n    s_boxes=segmentation.boxes.xyxy.cpu().numpy()\n    s_classes=segmentation.boxes.cls.int().cpu().numpy()\n    s_polygons=[] if segmentation.masks is None else segmentation.masks.xy\n    used=set(); objects=[]\n    for bb,c,conf in zip(d_boxes,d_classes,d_scores):\n        options=[(overlap(bb,sb),j) for j,(sb,sc) in enumerate(zip(s_boxes,s_classes))\n                 if sc==c and j not in used]\n        score,j=max(options,default=(0.,-1))\n        is_match=(j>=0 and score>=0.50 and j<len(s_polygons) and len(s_polygons[j])>=3)\n        polygon=None; centroid=None\n        if is_match:\n            used.add(j)\n            pts=np.asarray(s_polygons[j],dtype=np.float32)\n            mom=cv2.moments(pts)\n            if mom['m00']>1e-12:\n                centroid=[round(float(mom['m10']/mom['m00']),1),\n                          round(float(mom['m01']/mom['m00']),1)]\n            polygon=np.round(pts,1).tolist()\n        objects.append({'material':CONFIG['class_names'][int(c)],\n             'confidence':round(float(conf),4),\n             'bbox_xyxy':[round(float(v),1) for v in bb],\n             'mask_polygon_xy':polygon,'pick_point_xy':centroid,\n             'requires_review':not is_match})\n    counts={name:sum(o['material']==name for o in objects) for name in CONFIG['class_names']}\n    return {'image_width':img.width,'image_height':img.height,'objects':objects,\n            'counts':counts,'pick_point_units':'original_image_pixels'}\n"
(service_dir/'service.py').write_text(service_source,encoding='utf-8')
(service_dir/'requirements.txt').write_text(
    '\n'.join(f'{n}=={version(n)}' for n in ['ultralytics','torch','fastapi','uvicorn',
       'python-multipart','numpy','pillow','opencv-python','httpx'])+'\n')
(service_dir/'README.txt').write_text(
    'cd to this folder and run: uvicorn service:app --host 0.0.0.0 --port 8000\n'
    'POST image: curl -F file=@image.PNG http://localhost:8000/predict\n'
    'GET health: curl http://localhost:8000/health\n'
    'This Colab directory is not a permanent public deployment.\n')
print('Service generated:',service_dir)
print('Weights verified by SHA256 at startup; release manifest in Drive')
# Draft model card; review and update it before publishing.
card_path = service_dir/'MODEL_CARD_DRAFT.md'
card_text = (
    '# Project 9 — Model Card (DRAFT)\n\n'
    '## Purpose\nPrototype waste recognition on ZeroWaste-f conveyor images: four material labels, per-object polygons and mask centroids.\n\n'
    '## Training/evidence\nM4 YOLO11n: 34 epochs, validation mAP50 ~0.4832 and mAP50–95 ~0.3245. '
    'M5 YOLO11n-seg: interrupted at 26 epochs, best recorded validation mask mAP50–95 ~0.2888. '
    'No final Test metrics have been verified.\n\n'
    '## Limitations\nOnly four ZeroWaste classes. Many unlabeled objects, strong class imbalance, and possible cross-domain failure; '
    'do not use as an autonomous industrial sorting controller or for hazardous-waste safety decisions. '
    'The conservative grouped candidate excluded 1,272 images. Centroid is not guaranteed to be physically graspable.\n\n'
    '## Status and review\nCPU service prototype, not continuously hosted. ONNX parity and hardware throughput must be measured. '
    'Team members must review all claims, class definitions and dataset licence before submission.\n'
)
card_path.write_text(card_text,encoding='utf-8')
print('Model card draft:',card_path)



## 16. M8 local HTTP benchmark — explicitly preliminary
Uses FastAPI `TestClient` (in-process HTTP), after an initial warm-up, to measure **end-to-end request-to-JSON** latency for one **Validation** image. It does not prove live external availability or concurrency performance. `serial_requests_per_sec` is not the same as physical conveyor throughput. CPU RSS sampled after requests is **not** a peak-memory measurement; benchmark on final deployment hardware again.


In [ ]:
import time,json,importlib.util,sys,os
import numpy as np
from fastapi.testclient import TestClient

assert (service_dir/'service.py').is_file()
spec=importlib.util.spec_from_file_location('project9_service_v3',service_dir/'service.py')
module=importlib.util.module_from_spec(spec)
sys.modules['project9_service_v3']=module
spec.loader.exec_module(module)
client=TestClient(module.app)
assert client.get('/health').status_code==200
sample=sorted((det_dir/'images/val').glob('*.PNG'))[0]
blob=sample.read_bytes()
def request_once():
    response=client.post('/predict',files={'file':(sample.name,blob,'image/png')})
    response.raise_for_status()
    return response
warmup=3; repetitions=25
for _ in range(warmup):request_once()
lat=[]
for _ in range(repetitions):
    tic=time.perf_counter();request_once();lat.append((time.perf_counter()-tic)*1000)
p50,p95,p99=np.percentile(lat,[50,95,99])
report={'split':'val','sample':sample.name,'warmup':warmup,'timed_calls':repetitions,
        'backend':'PyTorch','device':'cpu','input_bytes':len(blob),
        'p50_ms':float(p50),'p95_ms':float(p95),'p99_ms':float(p99),
        'serial_requests_per_sec':float(1000/np.mean(lat)),
        'limitations':'TestClient in-process, one frame, CPU, not peak memory or physical belt throughput'}
(service_dir/'http_benchmark_VAL_preliminary.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))


## 17. Open-world A (5 points): **Visual search** — runnable validation experiment
**Function:** a detected object can retrieve visually similar *training* objects as references for a human sorter. Compare a frozen ImageNet ResNet-18 feature encoder against an 8-bin-per-channel RGB histogram baseline. Each has a gallery of labelled crops from **grouped TRAIN** and queries from **grouped VAL** (no Test). Record material top-1 / hit@5, speed, storage and a failure example; confirm nearest images visually. This is **not run in this uploaded notebook**, so leave the report's numerical cells blank until executed. May work on CPU; GPU is faster.


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, time, json
from PIL import Image
from collections import defaultdict
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# Reproducible, class-balanced validation-only sampling of real annotated object boxes.
RNG=np.random.default_rng(42)
open_dir=DRIVE_PROJECT/'open_world'/'visual_search'
open_dir.mkdir(parents=True,exist_ok=True)

def choose_objects(split,per_class):
    obj=defaultdict(list)
    for label in sorted((det_dir/'labels'/split).glob('*.txt')):
        image=det_dir/'images'/split/(label.stem+'.PNG')
        if not image.is_file(): continue
        for line in label.read_text().splitlines():
            c,xc,yc,w,h=map(float,line.split())
            if w>0.003 and h>0.003:
                obj[int(c)].append((str(image),int(c),xc,yc,w,h))
    chosen=[]
    for c in range(len(CLASS_NAMES)):
        assert obj[c],f'Missing material class {c} in {split}'
        indices=RNG.choice(len(obj[c]),size=min(per_class,len(obj[c])),replace=False)
        chosen.extend(obj[c][int(i)] for i in indices)
    return chosen

def crop_object(row):
    path,c,xc,yc,w,h=row
    with Image.open(path) as img:
        img=img.convert('RGB');W,H=img.size
        x1=max(0,int((xc-w/2)*W)); x2=min(W,int(np.ceil((xc+w/2)*W)))
        y1=max(0,int((yc-h/2)*H)); y2=min(H,int(np.ceil((yc+h/2)*H)))
        return img.crop((x1,y1,max(x1+1,x2),max(y1+1,y2)))

def unit(x):
    return x/(np.linalg.norm(x,axis=-1,keepdims=True)+1e-10)

def rgb_hist(crops):
    out=[]
    for im in crops:
        a=np.asarray(im.resize((64,64)),dtype=np.uint8)
        feats=[np.histogram(a[:,:,channel],bins=8,range=(0,256))[0] for channel in range(3)]
        out.append(np.concatenate(feats))
    return unit(np.asarray(out,dtype=np.float32))

gallery=choose_objects('train',40)
queries=choose_objects('val',10)
G=[crop_object(x) for x in gallery]
Q=[crop_object(x) for x in queries]
gallery_y=np.array([x[1] for x in gallery]);query_y=np.array([x[1] for x in queries])

weights=ResNet18_Weights.DEFAULT
encoder=resnet18(weights=weights);encoder.fc=nn.Identity()
device='cuda' if torch.cuda.is_available() else 'cpu'
encoder=encoder.to(device).eval()
transform=weights.transforms()

def embed(crops,batch=16):
    vectors=[]
    with torch.inference_mode():
        for start in range(0,len(crops),batch):
            tensor=torch.stack([transform(x) for x in crops[start:start+batch]]).to(device)
            vectors.append(encoder(tensor).cpu().numpy())
    return unit(np.concatenate(vectors).astype(np.float32))

t0=time.perf_counter();g_feature=embed(G);q_feature=embed(Q)
feature_seconds=time.perf_counter()-t0
h_gallery=rgb_hist(G);h_query=rgb_hist(Q)

def retrieval_metrics(g,q):
    rankings=(q@g.T).argsort(axis=1)[:,::-1]
    top1=gallery_y[rankings[:,0]]
    top5=gallery_y[rankings[:,:5]]
    return {'top1_accuracy':float(np.mean(top1==query_y)),
            'hit_at_5':float(np.mean(np.any(top5==query_y[:,None],axis=1))),
            'precision_at_5':float(np.mean(top5==query_y[:,None]))},rankings

resnet_scores,ranking=retrieval_metrics(g_feature,q_feature)
hist_scores,_=retrieval_metrics(h_gallery,h_query)
np.savez_compressed(open_dir/'gallery_index_train.npz',embeddings=g_feature,
                    labels=gallery_y,source_paths=np.array([x[0] for x in gallery]))
(open_dir/'comparison_VAL.json').write_text(json.dumps({
    'encoder':'ImageNet ResNet18 frozen','baseline':'RGB histogram (8x3)',
    'split':'train gallery / val queries', 'gallery_size':len(gallery),
    'query_size':len(queries),'resnet':resnet_scores,'histogram':hist_scores,
    'embedding_seconds':feature_seconds,'index_bytes':int(g_feature.nbytes)},indent=2))
print('Visual search:',open_dir/'comparison_VAL.json')
print('Frozen ResNet18:',resnet_scores,'RGB histogram baseline:',hist_scores)
print('Example query and its 5 training nearest neighbors:',queries[0][0],
      [(CLASS_NAMES[int(gallery_y[i])],gallery[i][0]) for i in ranking[0,:5]])


## 17B. Open-world B (5 points): **Zero-shot CLIP material classification**
**Different topic from retrieval.** A text-image model predicts one of four material names for uncertain waste crops **without fine-tuning on the ZeroWaste training set**. Function: optional second opinion/relabeling suggestion for a manual reviewer; not a tested unknown-class detector. Compare 2 prompt wordings on **Validation** against the training-gallery RGB histogram nearest-neighbor baseline from Topic A, and log accuracy/macro-F1 and failures. Model download required. Use the *same actual cropped validation labels*, never invent unknown labels. A Test score must be generated only once after prompts are frozen. This section is **not yet executed**.


In [ ]:
%pip -q install transformers


In [ ]:
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score
import numpy as np,pandas as pd,json,torch

assert 'queries' in globals() and 'h_gallery' in globals(), 'Run open-world A first'
model_id='openai/clip-vit-base-patch32'
clip=CLIPModel.from_pretrained(model_id).eval().to(device)
processor=CLIPProcessor.from_pretrained(model_id)
material_names=['cardboard','metal','rigid plastic','soft plastic']
prompts={
  'plain':[f'a photo of {name}' for name in material_names],
  'conveyor':[f'a close-up photo of {name} waste on a recycling conveyor belt' for name in material_names]
}

result_rows=[]
for prompt_name,texts in prompts.items():
    logits=[]
    with torch.inference_mode():
        for i in range(0,len(Q),8):
            inputs=processor(text=texts,images=Q[i:i+8],return_tensors='pt',padding=True)
            inputs={k:v.to(device) for k,v in inputs.items()}
            logits.append(clip(**inputs).logits_per_image.detach().cpu().numpy())
    prediction=np.concatenate(logits).argmax(axis=1)
    result_rows.append({'method':'CLIP zero-shot','prompt':prompt_name,'split':'val',
        'accuracy':accuracy_score(query_y,prediction),
        'macro_f1':f1_score(query_y,prediction,labels=list(range(4)),average='macro',zero_division=0)})

# Baseline: nearest train crop based on normalized RGB histogram (no CLIP training).
hist_prediction=gallery_y[(h_query@h_gallery.T).argmax(axis=1)]
result_rows.append({'method':'RGB histogram NN baseline','prompt':'none','split':'val',
    'accuracy':accuracy_score(query_y,hist_prediction),
    'macro_f1':f1_score(query_y,hist_prediction,labels=list(range(4)),average='macro',zero_division=0)})
openB_dir=DRIVE_PROJECT/'open_world'/'zero_shot_clip'
openB_dir.mkdir(parents=True,exist_ok=True)
results=pd.DataFrame(result_rows)
results.to_csv(openB_dir/'CLIP_vs_histogram_VAL.csv',index=False)
print(results.round(4).to_string(index=False))
print('Valid conclusions: zero-shot 4-known-material accuracy ONLY; not unknown detection')


> **HISTORICAL SNAPSHOT ONLY.** M5 later completed early stopping at epoch 36 and the original Test evaluation has been executed. Do not follow the old resume or Test instructions.

## 18. Freeze the final evaluation (Test) — **OFF by default**
**Do not run on Test yet.** Before the one final evaluation: finish M5 resume (or explicitly document why impossible), choose model/thresholds on Validation only, record acceptance criteria set *before* looking at Test, settle any baseline comparisons, and have all five members review the final metrics and code. The flag below defaults to `False` and writes only when deliberately enabled. This cell is an evaluation guard, not a completed test result. Save `best.pt`, exact `split_manifest.csv` and release-manifest hashes.


In [ ]:
raise RuntimeError('ARCHIVED code for provenance only; training / resume / frozen Test must not be rerun.')

RUN_FINAL_TEST = False  # never change to True until decisions are frozen and team reviewed

if RUN_FINAL_TEST:
    import json,hashlib
    def sha256_file(path):
        digest=hashlib.sha256()
        with open(path,'rb') as f:
            for data in iter(lambda:f.read(4*1024*1024),b''):
                digest.update(data)
        return digest.hexdigest()
    from ultralytics import YOLO
    assert (det_dir/'data.yaml').is_file() and (seg_dir/'data.yaml').is_file()
    assert M4_BEST.is_file() and M5_BEST.is_file()
    final_dir=DRIVE_PROJECT/'evaluation'/'FINAL_TEST_ONCE'
    if final_dir.exists():
        raise RuntimeError('Final test folder already exists. Do NOT silently repeat the Test evaluation.')
    final_dir.mkdir(parents=True)
    protocol={'confidence':0.25,'nms_iou':0.50,'imgsz':640,
       'split':'test','manifest_sha256':sha,
       'm4_checkpoint_sha256':sha256_file(M4_BEST),
       'm5_checkpoint_sha256':sha256_file(M5_BEST),
       'warning':'Both masks/boxes evaluated at low confidence for AP; operating threshold for deployment is 0.25'}
    (final_dir/'frozen_protocol.json').write_text(json.dumps(protocol,indent=2))
    try:
        m4_result=YOLO(str(M4_BEST)).val(data=str(det_dir/'data.yaml'),split='test',
            imgsz=640,conf=0.001,iou=0.5,plots=True,project=str(final_dir),name='M4')
        m5_result=YOLO(str(M5_BEST)).val(data=str(seg_dir/'data.yaml'),split='test',
            imgsz=640,conf=0.001,iou=0.5,plots=True,project=str(final_dir),name='M5')
        measured={'M4_mAP50':float(m4_result.box.map50),'M4_mAP50_95':float(m4_result.box.map),
             'M5_mask_mAP50':float(m5_result.seg.map50),'M5_mask_mAP50_95':float(m5_result.seg.map)}
        (final_dir/'results.json').write_text(json.dumps(measured,indent=2))
        print('FINAL ONE-TIME TEST:',measured)
    except Exception:
        print('Partial final Test artifacts exist; investigate before attempting any rerun')
        raise
else:
    print('FINAL TEST NOT RUN. This is intentional; keep Test unseen.')


> **HISTORICAL SNAPSHOT ONLY.** M5 later completed early stopping at epoch 36 and the original Test evaluation has been executed. Do not follow the old resume or Test instructions.

## Submission evidence and known limitations (not yet all finished)
- **M4 real:** 34 epochs completed; saved PyTorch checkpoint, validation AP and corrected threshold sweep. Still needed: compare a baseline, FP/FN gallery, per-class detection error analysis, video-like stream example, and final once-only Test result.
- **M5 real but interrupted:** best val mask mAP50–95 ~0.2888 at epoch 21, completed through epoch 26. Resume if GPU returns; compute full validation mask AP, separate mask IoU/Dice audit, compare baseline and show failures. Train/val mask audit visualized. No final Test value exists.
- **M8 code-only (not yet executed):** model/manifest packaging, ONNX export, small validation parity audit, locally runnable FastAPI, CPU prototype benchmark. Hosted service, full same-deployment benchmarks including memory, p50/p95/p99, throughput at actual belt speed, end-to-end per-stage timing remain to be demonstrated.
- **Open-world two topics:** runnable experiments added for image retrieval and zero-shot CLIP; not yet executed. Each requires a baseline comparison, meaningful *system* demonstration, failure cases, frozen settings and a once-only Test evaluation.
- **Split limitation:** 1,272 frames were excluded conservatively by filename-derived sequence grouping; explain lost training data, class imbalance, and remaining cross-sequence visual similarity risk. The current original Test set was preserved, but do not claim optical duplicate-freedom based solely on filename prefixes.
- **Acceptance criteria / report:** write pre-result thresholds or disclose if criteria were set after Phase 1; use the official 5-member Overleaf template; replace every placeholder; include real repo commit/W&B URLs; ensure every team member checks all claims against code. No original data or secrets in public GitHub.
